In [1]:

import numpy as np
import pandas as pd
import os
import PIL
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from sklearn import svm
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from PIL import Image
from sklearn.metrics import accuracy_score, multilabel_confusion_matrix
from torchvision.models import resnet18, ResNet18_Weights

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
DATASET_PATH = '/content/drive/MyDrive/models/'

data = {'img': [], 'models': []}

for model in ['infinix', 'honor', 'samsung', 'realme', 'xiaomi', 'iphone', 'tecno']:
    for dirname, _, filenames in os.walk(DATASET_PATH + model + '/'):
        for filename in filenames:
            data['img'].append(os.path.join(dirname, filename))
            data['models'].append(model)

df = pd.DataFrame.from_dict(data)

In [4]:
df

,img,models
0,/content/drive/MyDrive/models/infinix/item_751...,infinix
1,/content/drive/MyDrive/models/infinix/item_752...,infinix
2,/content/drive/MyDrive/models/infinix/item_752...,infinix
3,/content/drive/MyDrive/models/infinix/item_752...,infinix
4,/content/drive/MyDrive/models/infinix/item_753...,infinix
...,...,...
7009,/content/drive/MyDrive/models/tecno/item_73650...,tecno
7010,/content/drive/MyDrive/models/tecno/item_73578...,tecno
7011,/content/drive/MyDrive/models/tecno/item_73480...,tecno
7012,/content/drive/MyDrive/models/tecno/item_73520...,tecno


In [5]:
models_labels_map = {}

unique_models = df['models'].unique()
for i in range(len(unique_models)):
    models_labels_map[unique_models[i]] = i

In [6]:
df['models_label'] = df['models'].apply(lambda x: models_labels_map[x])
df

,img,models,models_label
0,/content/drive/MyDrive/models/infinix/item_751...,infinix,0
1,/content/drive/MyDrive/models/infinix/item_752...,infinix,0
2,/content/drive/MyDrive/models/infinix/item_752...,infinix,0
3,/content/drive/MyDrive/models/infinix/item_752...,infinix,0
4,/content/drive/MyDrive/models/infinix/item_753...,infinix,0
...,...,...,...
7009,/content/drive/MyDrive/models/tecno/item_73650...,tecno,6
7010,/content/drive/MyDrive/models/tecno/item_73578...,tecno,6
7011,/content/drive/MyDrive/models/tecno/item_73480...,tecno,6
7012,/content/drive/MyDrive/models/tecno/item_73520...,tecno,6


In [7]:
class Dataset(Dataset):
    def __init__(self, data, transform):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image = PIL.Image.open(self.data.loc[idx, "img"]).convert('RGB')
        image = self.transform(image)
        label = torch.tensor(self.data.loc[idx, "models_label"])
        return image, label

In [8]:
X_train, X_test, y_train, y_test = train_test_split(df["img"], df["models_label"], random_state=42)

X_train.index = np.arange(len(X_train))
y_train.index = np.arange(len(y_train))
X_test.index = np.arange(len(X_test))
y_test.index = np.arange(len(y_test))

In [9]:
transform_test = v2.Compose([
    v2.Resize(size=(224, 224)),
    v2.PILToTensor(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [10]:
train_ds = Dataset(pd.concat([X_train, y_train], axis=1), transform_test)
test_ds = Dataset(pd.concat([X_test, y_test], axis=1), transform_test)

train_loader = DataLoader(train_ds, batch_size=32, num_workers=4)
test_loader = DataLoader(test_ds, batch_size=32, num_workers=4)

In [11]:
class Extraction:
    def __init__(self, network):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.network = network.eval().to(self.device)

    def extract(self, loader):
        data_tmp = []
        label_tmp = []

        with torch.no_grad():
            for x, y in loader:
                x = x.to(self.device)

                outputs = self.network(x)
                data_tmp.append(outputs.view(-1, 512).cpu().numpy())

                label_tmp.append(y.cpu().numpy())

        return np.vstack(data_tmp), np.hstack(label_tmp)

In [12]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)
feature_extractor = torch.nn.Sequential(*list(model.children())[:-1])

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 200MB/s]


In [13]:
feature_extractor

Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Con

In [14]:
ext = Extraction(feature_extractor)

train_feature, train_label = ext.extract(train_loader)
test_feature, test_label = ext.extract(test_loader)

In [15]:
param = {
    # "kernel": ["rbf"],
    #  "C": [0.01, 0.1, 0.5],
    # "gamma": ["scale"]],
     "kernel": ["linear"],
    "C": [0.005, 0.01, 0.1, 1, 10]
}

svc = svm.SVC()
svm_grid = GridSearchCV(svc, param_grid=param, verbose=2, n_jobs=-1)

svm_grid.fit(train_feature, train_label)

Fitting 5 folds for each of 5 candidates, totalling 25 fits


GridSearchCV(estimator=SVC(), n_jobs=-1,
             param_grid={'C': [0.005, 0.01, 0.1, 1, 10], 'kernel': ['linear']},
             verbose=2)

In [16]:
print(svm_grid.best_params_)

{'C': 0.005, 'kernel': 'linear'}


In [17]:
y_pred_svm_grid = svm_grid.predict(test_feature)

In [20]:
print("Test accuracy:", accuracy_score(test_label, y_pred_svm_grid))

Test accuracy: 0.7115165336374002


In [19]:
train_pred = svm_grid.predict(train_feature)
print("Train accuracy:", accuracy_score(train_label, train_pred))

Train accuracy: 0.7977186311787072
